In [ ]:
import json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from pathlib import Path
from scipy.special import expit
from scipy import stats

import shap
import joblib

from sklearn.metrics import (
    roc_auc_score, average_precision_score, brier_score_loss,
    precision_recall_curve, classification_report,
    precision_score, recall_score, confusion_matrix,
    RocCurveDisplay, PrecisionRecallDisplay,
)
from sklearn.inspection import PartialDependenceDisplay

warnings.filterwarnings("ignore")

ARTIFACTS = Path("model_artifacts")
SEED = 42
TARGET_RECALL = 0.75   # ← change this single value to sweep recall targets
rng = np.random.default_rng(SEED)

# ── Load artifacts ────────────────────────────────────────────────────────────
m1  = joblib.load(ARTIFACTS / "m1_lr_baseline.pkl")
m2a = joblib.load(ARTIFACTS / "m2a_lr_era.pkl")

m2b_map = np.load(ARTIFACTS / "m2b_pymc_map.npz")
m3_post = np.load(ARTIFACTS / "m3_multilevel_advi.npz")

prep_common = joblib.load(ARTIFACTS / "preprocessor_common.pkl")
prep_m3     = joblib.load(ARTIFACTS / "preprocessor_m3.pkl")

arrs = np.load(ARTIFACTS / "test_arrays.npz")
X_test_sk        = arrs["X_test_sk"]
X_test_m3        = arrs["X_test_m3"]
y_test           = arrs["y_test"]
era_test         = arrs["era_test"]
school_test      = arrs["school_test"]
X_train_bg       = arrs["X_train_sk_sample"]   # background for SHAP
y_train          = arrs["y_train"]
era_train        = arrs["era_train"]

with open(ARTIFACTS / "feature_names.json") as f:
    fnames = json.load(f)
feat_common  = fnames["common"]
feat_m3      = fnames["m3"]
era_labels   = fnames["era_labels"]
school_labels = fnames["school_labels"]

# Era-augmented test matrix for M2a
X_test_m2a = np.column_stack([X_test_sk, era_test])
X_train_bg_m2a = np.column_stack([X_train_bg, era_train[:2000]])

# Pre-compute all probabilities
prob_m1  = m1.predict_proba(X_test_sk)[:, 1]
prob_m2a = m2a.predict_proba(X_test_m2a)[:, 1]
prob_m2b = expit(
    m2b_map["alpha"][era_test] + X_test_sk @ m2b_map["beta"]
)
prob_m3 = expit(
    m3_post["alpha_school"][school_test]
    + float(m3_post["beta_era"]) * era_test
    + X_test_m3 @ m3_post["beta"]
)

model_probs = {
    "M1 – Baseline LR":           prob_m1,
    "M2a – LR + era (sklearn)":   prob_m2a,
    "M2b – PyMC MAP":             prob_m2b,
    "M3 – Multilevel ADVI":       prob_m3,
}

print("Artifacts loaded. Test set:", X_test_sk.shape, "| dropout rate:", y_test.mean().round(4))


---
## 3.1  Test-Set Evaluation with Confidence Intervals

We evaluate all four models on the held-out test set (20 % of the data, stratified by era × dropout).
Bootstrap confidence intervals (n = 1 000 resamples) are reported for ROC-AUC, PR-AUC, and Brier score.
The classification threshold is chosen automatically to achieve ≥ 0.75 dropout recall.


In [ ]:
def find_recall_threshold(y_true, y_prob, target_recall=0.75):
    precision, recall, thresholds = precision_recall_curve(y_true, y_prob)
    best_t = thresholds[0]
    for p, r, t in zip(precision[:-1], recall[:-1], thresholds):
        if r >= target_recall:
            best_t = t
    return best_t

def bootstrap_metrics(y_true, y_prob, n_boot=1000, seed=42):
    rng_b = np.random.default_rng(seed)
    aucs, aps, briers = [], [], []
    n = len(y_true)
    for _ in range(n_boot):
        idx = rng_b.integers(0, n, n)
        if y_true[idx].sum() == 0:
            continue
        aucs.append(roc_auc_score(y_true[idx], y_prob[idx]))
        aps.append(average_precision_score(y_true[idx], y_prob[idx]))
        briers.append(brier_score_loss(y_true[idx], y_prob[idx]))
    def ci(arr):
        return np.percentile(arr, [2.5, 97.5])
    return {
        "roc_auc":  (roc_auc_score(y_true, y_prob), *ci(aucs)),
        "pr_auc":   (average_precision_score(y_true, y_prob), *ci(aps)),
        "brier":    (brier_score_loss(y_true, y_prob), *ci(briers)),
    }

rows = []
for name, prob in model_probs.items():
    thr = find_recall_threshold(y_test, prob, TARGET_RECALL)
    y_pred = (prob >= thr).astype(int)
    m = bootstrap_metrics(y_test, prob)
    rows.append({
        "Model": name,
        "ROC-AUC": f"{m['roc_auc'][0]:.4f} [{m['roc_auc'][1]:.4f}–{m['roc_auc'][2]:.4f}]",
        "PR-AUC":  f"{m['pr_auc'][0]:.4f} [{m['pr_auc'][1]:.4f}–{m['pr_auc'][2]:.4f}]",
        "Brier":   f"{m['brier'][0]:.4f} [{m['brier'][1]:.4f}–{m['brier'][2]:.4f}]",
        "Threshold": f"{thr:.3f}",
        "Recall":  f"{recall_score(y_test, y_pred):.3f}",
        "Precision": f"{precision_score(y_test, y_pred, zero_division=0):.3f}",
    })

ci_df = pd.DataFrame(rows)
display(ci_df.set_index("Model"))
print(f"\nNote: threshold chosen to achieve recall \u2265 {TARGET_RECALL:.0%}")
ci_df.to_csv("ci_metrics.csv", index=False)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for name, prob in model_probs.items():
    RocCurveDisplay.from_predictions(y_test, prob, name=name, ax=axes[0])
    PrecisionRecallDisplay.from_predictions(y_test, prob, name=name, ax=axes[1])

axes[0].set_title("ROC Curves — Test Set")
axes[0].legend(fontsize=8)
axes[1].set_title("Precision-Recall Curves — Test Set\n(more informative for imbalanced data)")
axes[1].legend(fontsize=8, loc="upper right")
fig.tight_layout()
plt.savefig("fig_roc_pr.png", dpi=150)
plt.show()


> **📝 Interpretation prompt — ROC & PR curves**
>
> Answer the following questions in your report:
> 1. Which model achieves the highest PR-AUC? By how many points does it beat the baseline?
>    Is this difference practically meaningful given the CIs?
> 2. The PR baseline (random classifier) equals the class prevalence (~8 %). How far above this
>    baseline do the best models sit at recall = 0.75?
> 3. The ROC curves cluster tightly (0.71–0.75 AUC). What does this tell you about the marginal
>    value of adding era/school structure versus raw features for discriminating dropouts?
> 4. Compare ROC-AUC vs PR-AUC as evaluation criteria for this use case (8 % prevalence).
>    Which should be the primary metric for an early-warning system and why?


---
## 3.2  Interpretability Analysis

### 3.2.1  Logistic Regression — Coefficients (M1 & M2a)


In [ ]:
# Bootstrap coefficient CIs for M1
n_boot = 500
coef_boot = []
rng_b = np.random.default_rng(SEED)
n = len(y_train)
# NOTE: refitting 500 times is slow; use the stored coef + Hessian approximation instead.
# We use a Normal approximation: se ≈ 1 / sqrt(n * p * (1-p)) * scale factor from sklearn.
# For a proper CI use bootstrap on the training set if compute allows.
# Here we report point estimates + rank ordering, flagging the top-20 by |coef|.

coef_series = pd.Series(m1.coef_[0], index=feat_common).sort_values(key=abs, ascending=False)
top_coefs = coef_series.head(20)

fig, ax = plt.subplots(figsize=(8, 6))
colors = ["#d62728" if v > 0 else "#1f77b4" for v in top_coefs.values]
ax.barh(top_coefs.index[::-1], top_coefs.values[::-1], color=colors[::-1])
ax.axvline(0, color="black", lw=0.8)
ax.set_xlabel("Coefficient (log-odds scale, standardised features)")
ax.set_title("M1 – Top 20 Features by |Coefficient|\n(red = risk factor, blue = protective)")
fig.tight_layout()
plt.savefig("fig_lr_coefs.png", dpi=150)
plt.show()

# Era coefficient from M2a
era_coef = m2a.coef_[0, -1]
print(f"\nM2a – era_code coefficient: {era_coef:.4f}  (OR = {np.exp(era_coef):.3f})")
print("Tec21 is associated with", "HIGHER" if era_coef > 0 else "LOWER", "dropout log-odds")


> **📝 Interpretation prompt — LR Coefficients**
>
> 1. The top positive coefficients (red) represent **risk factors** for dropout. Name the top 3
>    and explain their substantive meaning in the TEC context. Are they consistent with prior
>    theory on student retention (e.g., Tinto's model)?
> 2. The top negative coefficients (blue) represent **protective factors**. Interpret the
>    scholarship-related features: what do they reveal about the role of financial support?
> 3. The `activity_missing_flag` cluster appears with a large positive coefficient. Is this
>    a genuine risk signal or a data-quality artefact? What would you recommend?
> 4. M2a adds `era_code` as a fixed effect. The coefficient is **-0.1379 (OR ≈ 0.87)**.
>    Interpret this: does the Tec21 reform appear to reduce dropout risk after controlling
>    for individual covariates? What are the causal identification concerns?


### 3.2.2  SHAP Values — Global & Local Explanations (M1)


In [ ]:
# Linear SHAP explainer is exact and fast for logistic regression
explainer_m1 = shap.LinearExplainer(m1, X_train_bg, feature_names=feat_common)
shap_vals_m1 = explainer_m1(X_test_sk)

# ── SHAP Beeswarm ────────────────────────────────────────────────────────────
# NOTE: Do NOT pre-create fig/ax — SHAP beeswarm manages its own figure.
# Calling plt.subplots() before shap.plots.beeswarm() causes the plot to
# render into empty axes (known shap + matplotlib conflict).
plt.close("all")
shap.plots.beeswarm(shap_vals_m1, max_display=20, show=False)
plt.title("SHAP Beeswarm — M1 Baseline LR (global feature importance)")
plt.tight_layout()
plt.savefig("fig_shap_beeswarm.png", dpi=150, bbox_inches="tight")
plt.show()
plt.close("all")


In [ ]:
plt.close("all")
shap.plots.bar(shap_vals_m1, max_display=15, show=False)
plt.title("SHAP Mean |value| — Global importance")
plt.tight_layout()
plt.savefig("fig_shap_bar.png", dpi=150, bbox_inches="tight")
plt.show()
plt.close("all")

top_idx = int(np.argmax(prob_m1))
print(f"Explaining observation {top_idx}: predicted P(dropout) = {prob_m1[top_idx]:.3f}")
plt.close("all")
shap.plots.waterfall(shap_vals_m1[top_idx], max_display=12, show=False)
plt.title(f"SHAP Waterfall — highest-risk student (idx={top_idx})")
plt.tight_layout()
plt.savefig("fig_shap_waterfall.png", dpi=150, bbox_inches="tight")
plt.show()
plt.close("all")


> **📝 Interpretation prompt — SHAP values**
>
> 1. **Beeswarm**: The horizontal axis is SHAP value (impact on log-odds). Identify the feature
>    with the widest spread. What does a wide spread indicate about heterogeneity in its effect
>    across students?
> 2. Do the SHAP rankings agree with the raw coefficient rankings from Cell 6? If they differ,
>    explain why SHAP can give a different picture even for a linear model (hint: feature
>    correlation and base-rate effects).
> 3. **Waterfall (individual student)**: Walk through the 3 most influential features pushing
>    this student toward dropout. Are these features the institution could act on
>    (actionable) or fixed characteristics (non-actionable)?
> 4. Connect these findings to your research question: **what have you learned about the
>    mechanisms of dropout** at TEC? Are the top features academic, socioeconomic, or
>    institutional in nature?


### 3.2.3  Partial Dependence Plots — Top Variables


In [ ]:
# Identify top-5 numeric features by mean |SHAP| value
mean_abs_shap = np.abs(shap_vals_m1.values).mean(axis=0)
top5_idx = np.argsort(mean_abs_shap)[::-1][:5]
top5_names = [feat_common[i] for i in top5_idx]
print("Top 5 features for PDP:", top5_names)

X_eval = X_test_sk.astype("float64")   # sklearn internals expect float64

fig, axes = plt.subplots(1, len(top5_idx), figsize=(4 * len(top5_idx), 4), sharey=False)

for ax, feat_i, name in zip(axes, top5_idx.tolist(), top5_names):
    col = X_eval[:, feat_i]
    uniq = np.unique(col)
    grid = uniq if len(uniq) <= 10 else np.linspace(np.percentile(col, 5), np.percentile(col, 95), 50)
    avg_preds = [
        m1.predict_proba(np.where(np.arange(X_eval.shape[1]) == feat_i, v, X_eval))[:, 1].mean()
        for v in grid
    ]
    marker = "o" if len(grid) <= 10 else None
    ax.plot(grid, avg_preds, color="#1f77b4", marker=marker, ms=4)
    ax.set_title(name, fontsize=9)
    ax.set_xlabel(name, fontsize=8)
    ax.grid(alpha=0.3)

fig.suptitle("Partial Dependence Plots — M1 Baseline LR", y=1.02)
fig.tight_layout()
plt.savefig("fig_pdp.png", dpi=150, bbox_inches="tight")
plt.show()


> **📝 Interpretation prompt — Partial Dependence Plots**
>
> 1. For the most important numeric feature, describe the shape of the PDP curve:
>    is the effect monotone, non-linear, or flat? What substantive explanation would you offer?
> 2. PDPs show the **marginal** effect averaging over all other features. What assumption does
>    this require, and when might it be violated in your dataset
>    (hint: think about era-specific features)?
> 3. Compare the direction of the PDP slope to the sign of the coefficient in Cell 6.
>    Do they agree? (They should for a linear model — confirm and explain.)
> 4. If you were advising the TEC retention office, which of the top-5 variables would you
>    highlight as the most **actionable** early warning signal and why?


---
## 3.2.4  ADVI Convergence Diagnostic (M3 — Multilevel Model)

ADVI (Automatic Differentiation Variational Inference) optimises a lower bound (ELBO) on the
log-marginal likelihood. If the ELBO has not plateaued by the final iteration, the variational
approximation is not converged and posterior means are unreliable.

We plot the full ELBO trace and compute the relative improvement in the last 20 % of iterations
to quantify convergence.


In [ ]:
# ── Load ELBO history ─────────────────────────────────────────────────────────
elbo_path = ARTIFACTS / "advi_elbo_history_m3.npy"

if elbo_path.exists():
    elbo_hist = np.load(elbo_path)
    n_iter = len(elbo_hist)

    # Smooth with a rolling window for readability
    window = max(1, n_iter // 100)
    elbo_smooth = pd.Series(elbo_hist).rolling(window, min_periods=1).mean().values

    # Convergence metric: relative ELBO change in last 20% of iterations
    cutoff = int(0.80 * n_iter)
    elbo_early  = elbo_smooth[cutoff]
    elbo_final  = elbo_smooth[-1]
    rel_change  = abs((elbo_final - elbo_early) / (abs(elbo_early) + 1e-8))

    fig, axes = plt.subplots(1, 2, figsize=(13, 4))

    # ── Full trace ───────────────────────────────────────────────────────────
    axes[0].plot(elbo_hist,   color="lightgrey", lw=0.4, label="raw")
    axes[0].plot(elbo_smooth, color="#1f77b4",   lw=1.5, label=f"smoothed (window={window})")
    axes[0].axvline(cutoff, color="red", ls="--", lw=0.8,
                    label=f"80 % mark (iter {cutoff:,})")
    axes[0].set_xlabel("ADVI Iteration")
    axes[0].set_ylabel("Negative ELBO (loss; lower = better)")
    axes[0].set_title("M3 — Full ADVI Loss Trace")
    axes[0].legend(fontsize=8)
    axes[0].grid(alpha=0.3)

    # ── Last 20 % zoom ───────────────────────────────────────────────────────
    axes[1].plot(np.arange(cutoff, n_iter), elbo_hist[cutoff:],
                 color="lightgrey", lw=0.4, label="raw")
    axes[1].plot(np.arange(cutoff, n_iter), elbo_smooth[cutoff:],
                 color="#d62728", lw=1.5, label="smoothed")
    axes[1].set_xlabel("ADVI Iteration")
    axes[1].set_title(f"Last 20 % of iterations (zoom)\nRelative change: {rel_change:.4%}")
    axes[1].legend(fontsize=8)
    axes[1].grid(alpha=0.3)

    fig.tight_layout()
    plt.savefig("fig_advi_elbo.png", dpi=150)
    plt.show()

    # ── Convergence summary ──────────────────────────────────────────────────
    print(f"\n── ADVI Convergence Summary ─────────────────────────────────────")
    print(f"  Total iterations    : {n_iter:,}")
    print(f"  ELBO at iter 1      : {elbo_hist[0]:.1f}")
    if n_iter >= 5000:
        print(f"  ELBO at iter 5,000  : {elbo_hist[4999]:.1f}  (original stopping point)")
        print(f"  ELBO change 1\u21925k    : {elbo_hist[4999] - elbo_hist[0]:+.1f}")
    print(f"  ELBO at iter {n_iter:,}  : {elbo_hist[-1]:.1f}")
    if n_iter >= 5000:
        print(f"  ELBO change 5k\u2192{n_iter//1000}k  : {elbo_hist[-1] - elbo_hist[4999]:+.1f}")
    print(f"  Rel. change (last 20%): {rel_change:.4%}")
    print()
    if rel_change < 0.001:
        print("  \u2713 CONVERGED  \u2014 relative change < 0.1 % in final 20 % of iterations")
    elif rel_change < 0.01:
        print("  \u26a0 LIKELY CONVERGED  \u2014 relative change < 1 %; inspect zoom plot")
    else:
        print("  \u2717 NOT CONVERGED  \u2014 relative change > 1 %; consider more iterations")
        print("    \u2192 If plateau is visible but at high loss, mean-field ADVI may be too")
        print("      restrictive; consider full NUTS on a subsample instead.")

else:
    print("\u26a0 ELBO history file not found.")
    print("  Re-run the export cell in experiments.ipynb with the updated tracker code.")
    print(f"  Expected path: {elbo_path}")


> **📝 Interpretation prompt — ADVI Convergence**
>
> 1. **Full trace**: Does the loss curve show a clear elbow (rapid early drop, then plateau)?
>    At approximately which iteration does the curve flatten? What does this tell you about
>    the minimum number of iterations needed for this model?
>
> 2. **Zoom (last 20 %)**: Is the smoothed curve still declining, roughly flat, or noisy
>    around a plateau?
>    - Still declining → not converged; report M3 results with a caveat
>    - Flat but noisy → likely converged; noise is Monte Carlo variance in the ELBO estimator
>    - Smooth plateau → converged; results are reliable
>
> 3. **Comparing 5k vs 10k**: By how many ELBO units did the loss improve between iteration
>    5,000 and 10,000? Is this improvement meaningful relative to the total drop from
>    iteration 1 to 5,000?
>
> 4. **Implication for M3 performance**: M3 has a lower ROC-AUC than M1 (baseline) and an
>    implausibly low threshold (0.007). If the ELBO has converged, this suggests the
>    **mean-field ADVI approximation itself** is the bottleneck — not iteration count.
>    Mean-field ADVI assumes all parameters are independent, which is violated in a
>    multilevel model where school intercepts correlate with the global mean.
>    If the ELBO has *not* converged, the low threshold is simply an artefact of a
>    degenerate solution. Distinguish these two cases based on the plots.
>
> 5. **Recommendation**: Given what you observe, would you recommend (a) running more ADVI
>    iterations, (b) switching to full NUTS on a stratified subsample, or (c) dropping M3
>    from the final comparison? Justify your choice.


---
## 3.3  Robustness Analysis

We test whether conclusions are stable under alternative methodological choices.
A result is considered **robust** if the change in ROC-AUC is < 0.01 and in PR-AUC < 0.02. Recall target: 75%.


In [ ]:
thresholds_grid = np.linspace(0.05, 0.70, 60)

fig, axes = plt.subplots(2, 2, figsize=(13, 9))
axes = axes.ravel()

for ax, (name, prob) in zip(axes, model_probs.items()):
    precs, recs = [], []
    for t in thresholds_grid:
        y_pred_t = (prob >= t).astype(int)
        precs.append(precision_score(y_test, y_pred_t, zero_division=0))
        recs.append(recall_score(y_test, y_pred_t))
    ax.plot(thresholds_grid, recs,   label="Recall",    color="#1f77b4")
    ax.plot(thresholds_grid, precs,  label="Precision", color="#d62728")
    ax.axvline(find_recall_threshold(y_test, prob, TARGET_RECALL), ls="--", color="grey", lw=0.9,
               label=f"auto-threshold @recall={TARGET_RECALL}")
    ax.set_title(name, fontsize=9)
    ax.set_xlabel("Classification threshold")
    ax.legend(fontsize=8)
    ax.set_ylim(0, 1)
    ax.grid(alpha=0.3)

fig.suptitle("Precision / Recall vs Threshold — all models")
fig.tight_layout()
plt.savefig("fig_threshold_sensitivity.png", dpi=150)
plt.show()


In [ ]:
mask_pre  = era_test == 0
mask_tec  = era_test == 1

print(f"Pre-Tec21 test: {mask_pre.sum():,} rows | dropout {y_test[mask_pre].mean():.3%}")
print(f"Tec21    test:  {mask_tec.sum():,} rows | dropout {y_test[mask_tec].mean():.3%}\n")

sub_rows = []
for name, prob in model_probs.items():
    for era_name, mask in [("Pre-Tec21", mask_pre), ("Tec21", mask_tec)]:
        thr = find_recall_threshold(y_test, prob, TARGET_RECALL)
        sub_rows.append({
            "Model": name, "Era": era_name,
            "ROC-AUC": roc_auc_score(y_test[mask], prob[mask]),
            "PR-AUC":  average_precision_score(y_test[mask], prob[mask]),
            "Recall":  recall_score(y_test[mask], (prob[mask] >= thr).astype(int)),
            "Precision": precision_score(y_test[mask], (prob[mask] >= thr).astype(int), zero_division=0),
        })

sub_df = pd.DataFrame(sub_rows)
display(sub_df.pivot(index="Model", columns="Era", values=["ROC-AUC","PR-AUC","Recall","Precision"])
        .round(4))


In [ ]:
school_rows = []
for school_id, school_name in enumerate(school_labels):
    mask_s = school_test == school_id
    if mask_s.sum() < 30 or y_test[mask_s].sum() < 5:
        continue
    for name, prob in model_probs.items():
        thr = find_recall_threshold(y_test, prob, TARGET_RECALL)
        school_rows.append({
            "Model": name, "School": school_name,
            "n": int(mask_s.sum()),
            "dropout_rate": float(y_test[mask_s].mean()),
            "ROC-AUC": roc_auc_score(y_test[mask_s], prob[mask_s]),
            "PR-AUC": average_precision_score(y_test[mask_s], prob[mask_s]),
            "Recall": recall_score(y_test[mask_s], (prob[mask_s] >= thr).astype(int)),
        })

school_df = pd.DataFrame(school_rows)
display(school_df[school_df["Model"] == "M1 – Baseline LR"]
        .set_index("School")[["n","dropout_rate","ROC-AUC","PR-AUC","Recall"]].round(4))


In [ ]:
# Build the sensitivity table required by component3.md
base_m1_auc = roc_auc_score(y_test, prob_m1)
base_m1_ap  = average_precision_score(y_test, prob_m1)
base_thr    = find_recall_threshold(y_test, prob_m1, TARGET_RECALL)

sens_rows = [
    {
        "Variation": "Baseline (M1)",
        "Metric Base": f"AUC={base_m1_auc:.4f}",
        "Metric Varied": "—",
        "Δ AUC": "—",
        "Interpretation": "Reference point"
    },
    {
        "Variation": "+ era fixed effect (M2a vs M1)",
        "Metric Base": f"AUC={base_m1_auc:.4f}",
        "Metric Varied": f"AUC={roc_auc_score(y_test, prob_m2a):.4f}",
        "Δ AUC": f"{roc_auc_score(y_test, prob_m2a) - base_m1_auc:+.4f}",
        "Interpretation": "Era grouping adds near-zero discriminative lift"
    },
    {
        "Variation": "PyMC partial pooling (M2b MAP vs M2a)",
        "Metric Base": f"AUC={roc_auc_score(y_test, prob_m2a):.4f}",
        "Metric Varied": f"AUC={roc_auc_score(y_test, prob_m2b):.4f}",
        "Δ AUC": f"{roc_auc_score(y_test, prob_m2b) - roc_auc_score(y_test, prob_m2a):+.4f}",
        "Interpretation": "Bayesian shrinkage has negligible effect with only 2 era groups"
    },
    {
        "Variation": "Multilevel school+era ADVI (M3 vs M1)",
        "Metric Base": f"AUC={base_m1_auc:.4f}",
        "Metric Varied": f"AUC={roc_auc_score(y_test, prob_m3):.4f}",
        "Δ AUC": f"{roc_auc_score(y_test, prob_m3) - base_m1_auc:+.4f}",
        "Interpretation": "School random effects reduce AUC — ADVI may underfit at 5k iterations"
    },
    {
        "Variation": "Threshold 0.5 → auto @recall=0.75 (M1)",
        "Metric Base": f"Prec@0.5={precision_score(y_test,(prob_m1>=0.5).astype(int),zero_division=0):.3f}",
        "Metric Varied": f"Prec@thr={precision_score(y_test,(prob_m1>=base_thr).astype(int),zero_division=0):.3f}",
        "Δ AUC": "N/A (threshold)",
        "Interpretation": "Lowering threshold sharply trades precision for recall — expected trade-off"
    },
    {
        "Variation": "Era subgroup: Pre-Tec21 vs Tec21 (M1)",
        "Metric Base": f"AUC(Pre)={roc_auc_score(y_test[mask_pre], prob_m1[mask_pre]):.4f}",
        "Metric Varied": f"AUC(Tec)={roc_auc_score(y_test[mask_tec], prob_m1[mask_tec]):.4f}",
        "Δ AUC": f"{roc_auc_score(y_test[mask_tec],prob_m1[mask_tec])-roc_auc_score(y_test[mask_pre],prob_m1[mask_pre]):+.4f}",
        "Interpretation": (
        f"Material gap: AUC drops {roc_auc_score(y_test[mask_tec], prob_m1[mask_tec]) - roc_auc_score(y_test[mask_pre], prob_m1[mask_pre]):.4f} "
        f"on Tec21 vs Pre-Tec21. Model trained predominantly on Pre-Tec21 data ({mask_pre.sum():,} rows) "
        f"may not capture Tec21-specific dropout patterns. Gap exceeds robustness threshold (0.01)."
    )
    },
]

sens_df = pd.DataFrame(sens_rows)
display(sens_df.set_index("Variation"))
sens_df.to_csv("sensitivity_table.csv", index=False)


> **📝 Interpretation prompt — Robustness**
>
> 1. **Threshold sensitivity**: The auto-threshold for recall=0.75 yields precision ≈ 0.13–0.14.
>    This means ~6–7 false alarms for every true detection. In a TEC intervention context
>    (e.g., advisor outreach), is this acceptable? What cost ratio between false negatives
>    and false positives would you recommend, and why?
> 2. **Era subgroup**: If the model performs materially differently on Pre-Tec21 vs Tec21
>    students, what are the implications for deploying it in the current (Tec21) era?
>    Should separate models be trained per era?
> 3. **School subgroup**: Identify the school with the largest gap between its dropout rate
>    and the model's per-school AUC. What might explain this? (Consider sample size,
>    unobserved school-specific factors, data quality.)
> 4. **Overall robustness verdict**: Summarise in 2–3 sentences whether the conclusions
>    (which features matter, which model is best) are stable across the variations tested.
>    Be honest about which decisions *do* materially change results.


---
## 3.4  Error Analysis — False Positives & False Negatives

We inspect the systematic patterns among students the best-available sklearn model (M1) misclassifies.


In [ ]:
# Use M1 at the auto threshold
thr_m1 = find_recall_threshold(y_test, prob_m1, TARGET_RECALL)
y_pred_m1 = (prob_m1 >= thr_m1).astype(int)

# Rebuild the raw test feature matrix from the preprocessor's inverse is not available,
# so we work with the transformed matrix + feature names.
test_df = pd.DataFrame(X_test_sk, columns=feat_common)
test_df["y_true"]  = y_test
test_df["y_pred"]  = y_pred_m1
test_df["prob"]    = prob_m1
test_df["era"]     = era_test
test_df["school"]  = school_test

FP = test_df[(test_df["y_pred"] == 1) & (test_df["y_true"] == 0)]
FN = test_df[(test_df["y_pred"] == 0) & (test_df["y_true"] == 1)]
TP = test_df[(test_df["y_pred"] == 1) & (test_df["y_true"] == 1)]
TN = test_df[(test_df["y_pred"] == 0) & (test_df["y_true"] == 0)]

print(f"TP: {len(TP):,}  |  FP: {len(FP):,}  |  FN: {len(FN):,}  |  TN: {len(TN):,}")

# Compare numeric feature means across error groups
numeric_feats = feat_common[:10]   # first 10 are the numeric ones (StandardScaler order)
compare_df = pd.DataFrame({
    "TP (true dropout, caught)":   TP[numeric_feats].mean(),
    "FN (true dropout, missed)":   FN[numeric_feats].mean(),
    "FP (retained, flagged)":      FP[numeric_feats].mean(),
    "TN (retained, correct)":      TN[numeric_feats].mean(),
}).T.round(3)
display(compare_df)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Era distribution of errors
era_map = {0: "Pre-Tec21", 1: "Tec21"}
for ax, (group_name, group_df) in zip(axes, [("False Positives", FP), ("False Negatives", FN)]):
    era_counts = group_df["era"].map(era_map).value_counts()
    ax.bar(era_counts.index, era_counts.values, color=["#1f77b4","#ff7f0e"])
    ax.set_title(f"{group_name} by Era\n(n={len(group_df):,})")
    ax.set_ylabel("Count")

fig.tight_layout()
plt.savefig("fig_fp_fn_era.png", dpi=150)
plt.show()

# Probability distribution of FP vs TP (how confident is the model in wrong calls?)
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(FP["prob"], bins=40, alpha=0.6, label=f"False Positives (n={len(FP):,})", color="#d62728")
ax.hist(TP["prob"], bins=40, alpha=0.6, label=f"True Positives  (n={len(TP):,})", color="#2ca02c")
ax.axvline(thr_m1, color="black", ls="--", label=f"threshold={thr_m1:.3f}")
ax.set_xlabel("Predicted P(dropout)")
ax.set_title("Score distributions: TP vs FP")
ax.legend()
fig.tight_layout()
plt.savefig("fig_fp_score_dist.png", dpi=150)
plt.show()


In [ ]:
# ── Operational cost analysis: threshold vs workload ─────────────────────────
# At the chosen threshold, how many students does an advisor need to review
# per true positive found?

review_rows = []
for t_val in np.arange(0.30, 0.70, 0.02):
    y_pred_t = (prob_m1 >= t_val).astype(int)
    tp = ((y_pred_t == 1) & (y_test == 1)).sum()
    fp = ((y_pred_t == 1) & (y_test == 0)).sum()
    fn = ((y_pred_t == 0) & (y_test == 1)).sum()
    flagged = tp + fp
    if flagged == 0:
        continue
    review_rows.append({
        "Threshold":         round(t_val, 2),
        "Students flagged":  int(flagged),
        "True dropouts caught": int(tp),
        "False alarms":      int(fp),
        "Reviews per TP":    round(flagged / max(tp, 1), 1),
        "Recall":            round(tp / (tp + fn + 1e-9), 3),
    })

cost_df = pd.DataFrame(review_rows)

fig, ax1 = plt.subplots(figsize=(10, 4))
ax2 = ax1.twinx()

ax1.plot(cost_df["Threshold"], cost_df["Recall"],
         color="#1f77b4", lw=2, label="Recall")
ax1.set_ylabel("Recall", color="#1f77b4")
ax1.set_ylim(0, 1)

ax2.plot(cost_df["Threshold"], cost_df["Reviews per TP"],
         color="#d62728", lw=2, ls="--", label="Reviews per true positive")
ax2.set_ylabel("Reviews per true positive (workload)", color="#d62728")

ax1.axvline(find_recall_threshold(y_test, prob_m1, TARGET_RECALL),
            color="grey", ls=":", lw=1.2,
            label=f"auto-threshold @recall={TARGET_RECALL}")

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, fontsize=8, loc="center right")

ax1.set_xlabel("Classification threshold")
ax1.set_title("Recall vs Advisor Workload Trade-off (M1)\n"
              "Left axis: recall | Right axis: # students reviewed per true dropout found")
ax1.grid(alpha=0.3)
fig.tight_layout()
plt.savefig("fig_operational_cost.png", dpi=150)
plt.show()

# Print the row at the auto-threshold
auto_thr = find_recall_threshold(y_test, prob_m1, TARGET_RECALL)
closest = cost_df.iloc[(cost_df["Threshold"] - auto_thr).abs().argsort()[:1]]
print(f"\nAt auto-threshold \u2248 {auto_thr:.3f}:")
display(closest)


> **📝 Interpretation prompt — Error Analysis**
>
> **False Positives (students flagged as dropout risk who were retained):**
> 1. Looking at the numeric feature comparison table, which features are *highest* for FP
>    students compared to TN students? What type of student gets false-flagged — is there a
>    profile (e.g., low-income but resilient, or students with missing data)?
> 2. Are false positives disproportionately concentrated in one era or school?
>    What does this imply for equity in an automated warning system?
>
> **False Negatives (students who dropped out but were missed):**
> 3. What profile characterises the FN group compared to TP students?
>    Are these "quiet" dropouts — students who look academically fine but leave for
>    non-academic reasons (financial, personal)?
> 4. The model misses ~25 % of actual dropouts (recall target was 0.75). From an
>    institutional perspective, are these misses random or systematic? If systematic,
>    what additional data would you request to reduce them?
>
> **Overall:**
> 5. Do these error patterns reveal any **fairness concerns** (systematic disadvantage
>    to a subgroup)? Reference specific schools or eras if the data supports it.
> 6. Summarise in one paragraph: what are the **main limitations** of the current modelling
>    approach, and what methodological improvements would you prioritise for the next iteration?


---
## Summary Checklist

Before submitting, verify:

- [ ] All four models evaluated on test set with bootstrap 95 % CIs
- [ ] ROC + PR curves plotted and labelled
- [ ] `TARGET_RECALL` set to the value used throughout (currently 0.75)
- [ ] LR coefficients interpreted (odds ratios, substantive meaning)
- [ ] SHAP beeswarm renders correctly (not empty axes)
- [ ] SHAP bar + waterfall for at least one model
- [ ] PDP for top-5 features
- [ ] **ADVI convergence plot** inspected and verdict written (converged / not converged)
- [ ] Sensitivity table — era subgroup row filled with real interpretation, not placeholder
- [ ] Era + school subgroup analysis discussed
- [ ] Operational cost (reviews per TP) curve examined and threshold choice justified
- [ ] FP/FN profiles described qualitatively
- [ ] All interpretation prompts answered in report prose
- [ ] `sensitivity_table.csv` and `ci_metrics.csv` saved
- [ ] `fig_advi_elbo.png` included in report appendix
